# Evaluating Model Outputs

This notebook demonstrates how to obtain and use logprobs from the Completions API. Many of these examples are adapted from the ["Using logprobs" from the OpenAI Cookbook](https://cookbook.openai.com/examples/using_logprobs#0-imports-and-utils).


## Logprobs and Classification

The first thing to notice is that the Responses and Completions APIs can return logprobs. Not every model provider will return logprobs. Two key parameters to obtain logprobs are:

- `logprobs`: whether to retunr the log rpobabilities of the output tokens. If true, returns the logprobs of each output token returned in the content message.
- `top_logprobs`: An integer between 0 and 20 specifying the number of most likely tokens to return at each token position, each with an associated log probability. This parameter requires `logprobs = True`.

Log probabilities or logprobs are $log(p)$ where $p$ is the probability of a token occurring at a specific position based on the previous tokens in the context.

In [2]:
%load_ext dotenv
%dotenv ../../05_src/.env
%dotenv ../../05_src/.secrets
import sys
sys.path.append('../../05_src/')

In [3]:
from utils.clients import get_client
import numpy as np
import os
client = get_client()
MODEL = os.getenv('MODEL', 'gpt-4o-mini')

First, we establish a simple interface that we can use for our experiments.

In [4]:
# define a new function
def get_completion(
    input: list[dict[str, str]], # several inputs, as a list of dictionaries: string to string
    model: str = "gpt-4o-mini", # model is a string parameter with default value
    max_tokens=500,
    temperature=0,
    tools=None,
    logprobs=None,  # whether to return log probabilities of the output tokens or not. If true, returns the log probabilities of each output token returned in the content of message..
    top_logprobs=None,
) -> str: # outputs a string
    params = {
        "model": model,
        "input": input,
        "max_output_tokens": max_tokens,
        "temperature": temperature,
        "tools": tools,
        "include": ["message.output_text.logprobs"] if logprobs else [],
        "top_logprobs": top_logprobs,
    }
    if tools:
        params["tools"] = tools

    completion = client.responses.create(**params) # ** means this function will be called in the format it requires
    return completion

In [5]:
headlines = [
    # Sports vs. Business
    "Caitlin Clark Signs Record $28M Nike Shoe Deal",
    "Formula 1 Teams Threaten Strike Over Revenue Sharing Dispute",
    # Sports vs. Politics
    "NBA Players Boycott Games to Protest Police Brutality",
    "IOC Bans Russian Athletes from Paris Olympics Over Ukraine War",
    # Entertainment vs. Politics
    "Jon Stewart Returns to The Daily Show Ahead of Election Season",
    "Celebrities Flood Campaign Trail as Election Nears",
    # Crime vs. Business
    "FTX Founder Sam Bankman-Fried Sentenced to 25 Years in Fraud Case",
    "Pharmacy Chain Sued for Knowingly Fueling Opioid Crisis",
    # Art vs. Entertainment
    "Taylor Swift's 'Tortured Poets Department' Wins Album of the Year Grammy",
    "Kanye West Announces Surprise Art Exhibition in Venice",
    # Crime vs. Politics
    "Senator's Chief of Staff Arrested on Bribery Charges",
    # Sports / Crime / Entertainment (three-way split)
    "O.J. Simpson, NFL Star Acquitted of Murder, Dies at 76",
]

In [6]:
CLASSIFICATION_PROMPT = """You will be given a headline of a news article.
Classify the article into one of the following categories: Business,  Politics, Crime, Sports, Art, and Entertainment.
Return only the name of the category, and nothing else.
MAKE SURE your output is one of the six categories stated.
Article headline: {headline}"""

We can use the interface to obtain the classification that we requested. However, it does not show the logprobs of the different top options.

In [7]:
# cycling through each headline
for headline in headlines:
    print(f"\nHeadline: {headline}")
    response = get_completion(
        [{"role": "user", "content": CLASSIFICATION_PROMPT.format(headline=headline)}],
        model=MODEL,
    )
    print(f"Category: {response.output_text}\n")


Headline: Caitlin Clark Signs Record $28M Nike Shoe Deal
Category: Sports


Headline: Formula 1 Teams Threaten Strike Over Revenue Sharing Dispute
Category: Sports


Headline: NBA Players Boycott Games to Protest Police Brutality
Category: Sports


Headline: IOC Bans Russian Athletes from Paris Olympics Over Ukraine War
Category: Sports


Headline: Jon Stewart Returns to The Daily Show Ahead of Election Season
Category: Entertainment


Headline: Celebrities Flood Campaign Trail as Election Nears
Category: Politics


Headline: FTX Founder Sam Bankman-Fried Sentenced to 25 Years in Fraud Case
Category: Crime


Headline: Pharmacy Chain Sued for Knowingly Fueling Opioid Crisis
Category: Crime


Headline: Taylor Swift's 'Tortured Poets Department' Wins Album of the Year Grammy
Category: Entertainment


Headline: Kanye West Announces Surprise Art Exhibition in Venice
Category: Art


Headline: Senator's Chief of Staff Arrested on Bribery Charges
Category: Politics


Headline: O.J. Simpson, N

Showing the top two options with their log and linear probabilities.

In [ ]:
for headline in headlines:
    print(f"\nHeadline: {headline}")
    API_RESPONSE = get_completion(
        [{"role": "user", "content": CLASSIFICATION_PROMPT.format(headline=headline)}],
        model="gpt-4o-mini",
        logprobs=True,
        top_logprobs=2,
    )
    top_n_logprobs = API_RESPONSE.output[0].content[0].logprobs[0].top_logprobs
    output_content = ""
    for i, logprob in enumerate(top_n_logprobs, start=1):
        output_content += (
            f"Output token {i}: {logprob.token}, "
            f"logprobs: {logprob.logprob}, "
            f"linear probability: {np.round(np.exp(logprob.logprob)*100,2)}%\n"
        )
    print(output_content)
    print("\n")

    # logprob is concatonated probability, not of a single token


Headline: Caitlin Clark Signs Record $28M Nike Shoe Deal
Output token 1: Sports, logprobs: -0.252163, linear probability: 77.71%
Output token 2: Business, logprobs: -1.502163, linear probability: 22.26%




Headline: Formula 1 Teams Threaten Strike Over Revenue Sharing Dispute
Output token 1: Sports, logprobs: -2.5e-05, linear probability: 100.0%
Output token 2: Sport, logprobs: -11.000025, linear probability: 0.0%




Headline: NBA Players Boycott Games to Protest Police Brutality
Output token 1: Sports, logprobs: -0.000161, linear probability: 99.98%
Output token 2: Politics, logprobs: -8.75016, linear probability: 0.02%




Headline: IOC Bans Russian Athletes from Paris Olympics Over Ukraine War
Output token 1: Politics, logprobs: -0.25193, linear probability: 77.73%
Output token 2: Sports, logprobs: -1.50193, linear probability: 22.27%




Headline: Jon Stewart Returns to The Daily Show Ahead of Election Season
Output token 1: Entertainment, logprobs: -0.00034, linear probability:

In this classification task, we see the usefulness of logprobs: 
+ We can determine the degree to which a model is "sure" about a classification that it has proposed. 
+ Based on logprobs, we can set a threshold under which human assistance is needed. 
+ Alternatively, we can set the logic of our code to provide several options if the logprobs are within a threshold.